<a href="https://colab.research.google.com/github/Anast2007/CAT-2---DEEP-LEARNING/blob/main/Stock_Market_Price_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import kagglehub
path = kagglehub.dataset_download("jainilcoder/netflix-stock-price-prediction")

### Step 1: Locating and Loading the Dataset

Here is how we load the CSV data into a Pandas DataFrame:
* **`import os`** and **`import pandas as pd`**: Imports the necessary library for file path navigation and the Pandas data analysis library.
* **`os.listdir(path)`**: Lists files in the downloaded kagglehub directory.
* **`pd.read_csv(...)`**: Reads the CSV file containing the stock data and loads it into a DataFrame named `df`.
* **`display(df.head())`**: Shows the first 5 rows of our dataset to verify it loaded correctly.

In [ ]:
import os
import pandas as pd
import kagglehub

# Download dataset and define path inside this cell to avoid NameError
path = kagglehub.dataset_download("jainilcoder/netflix-stock-price-prediction")

# Find any CSV file inside the downloaded dataset path
csv_files = [f for f in os.listdir(path) if f.endswith('.csv')]
csv_path = os.path.join(path, csv_files[0])

# Load the CSV file into a pandas DataFrame
df = pd.read_csv(csv_path)

# Display the first 5 rows
display(df.head())

### Step 2: Cleaning the Data

We need to clean the dataset before applying any transformations:
* **`df.isnull().sum()`**: Checks if there are any missing (NaN) values in our columns.
* **`df.columns = df.columns.str.strip().str.lower()`**: Standardizes column names by removing extra spaces and converting them to lowercase.
* **`pd.to_datetime(...)`**: Converts the 'date' column into datetime objects, which is crucial for handling time-series stock data properly.
* **`df.sort_values(...)`**: Ensures the stock prices are ordered chronologically by date.

In [ ]:
# Check for missing values
print("Missing values in each column:")
print(df.isnull().sum())

# Standardize column names to lowercase and strip whitespaces
df.columns = df.columns.str.strip().str.lower()

# Convert the 'date' column to datetime type if it exists
if 'date' in df.columns:
    df['date'] = pd.to_datetime(df['date'])
    # Sort the dataset by date chronologically
    df = df.sort_values('date').reset_index(drop=True)

# Display data types and information
display(df.info())
display(df.head())

### Step 3: Normalizing the Stock Data

Stock prices and volume can have very different scales. We normalize them to a range between `0` and `1` using `MinMaxScaler`:
* **`from sklearn.preprocessing import MinMaxScaler`**: Imports the scaler tool.
* **`df.select_dtypes(...)`**: Automatically selects columns that contain numbers (like Open, High, Low, Close, Volume) to scale.
* **`scaler.fit_transform(...)`**: Calculates the minimum and maximum value for each column and scales the values to the range [0, 1].
* **`df_normalized`**: We keep a copy of our cleaned and scaled dataset for modeling.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# Get all numerical columns (excluding the date column)
numerical_cols = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
print("Numerical columns to scale:", numerical_cols)

# Initialize the MinMaxScaler
scaler = MinMaxScaler()

# Create a copy of our DataFrame to hold normalized values
df_normalized = df.copy()

# Fit and transform the numerical columns to [0, 1] range
df_normalized[numerical_cols] = scaler.fit_transform(df[numerical_cols])

# Display the final normalized dataset
display(df_normalized.head())

### Step 4: Generate Input-Output Sequences

To train predictive models (like LSTM or GRU), we must transform our 1D time-series data into 2D sequences (windows):
* **`window_size`**: The number of past days to look at (e.g., 10 days).
* **`X` (Inputs)**: A list containing sequences of 10 days of historical feature values.
* **`y` (Outputs)**: The target value we want to predict (typically the next day's 'close' price).

In [ ]:
import numpy as np

def create_sequences(data, window_size, target_col_idx):
    X = []
    y = []
    # Loop through the data to extract windows
    for i in range(len(data) - window_size):
        # Grab a window of historical data (e.g., day i to day i + window_size - 1)
        X.append(data[i : i + window_size])
        # Grab the target variable at the next step (day i + window_size)
        y.append(data[i + window_size, target_col_idx])
    return np.array(X), np.array(y)

# Define a suitable window size (e.g., 10 days of historical data)
WINDOW_SIZE = 10

# Exclude 'date' column as it's not a numerical feature for sequence generation
feature_cols = ['open', 'high', 'low', 'close', 'adj close', 'volume']
data_matrix = df_normalized[feature_cols].values

# Target index is for the 'close' price (index 3 in our feature list)
target_index = feature_cols.index('close')

# Generate the sequence datasets
X, y = create_sequences(data_matrix, WINDOW_SIZE, target_index)

print(f"Original dataset shape: {data_matrix.shape}")
print(f"Input sequences shape (X): {X.shape} (Samples, Window Size, Features)")
print(f"Output target shape (y): {y.shape} (Samples,)")

### Step 5: Select and Justify the Model

For daily stock price forecasting, we choose a **Long Short-Term Memory (LSTM)** network:
* **Why LSTM?** Stock data is non-linear and exhibits sequential dependencies over time. Unlike standard feedforward neural networks, LSTMs have loops that allow information to persist. They use forget, input, and output gates to track relevant long-term trends while disregarding short-term noise.
* **Architecture**: We will build a simple, clean LSTM network with an input layer matching our window size and feature count, followed by a `Dense` output layer predicting a single continuous value (the normalized closing price).

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from sklearn.model_selection import train_test_split

# Split data chronologically (train on past, validate on more recent data)
split_idx = int(len(X) * 0.8)
X_train, X_val = X[:split_idx], X[split_idx:]
y_train, y_val = y[:split_idx], y[split_idx:]

# Rebuild the LSTM model with a more stable architecture
model = Sequential([
    LSTM(64, return_sequences=False, input_shape=(X.shape[1], X.shape[2])),
    Dense(32, activation='linear'),
    Dense(1)
])

# Compile using Huber loss class explicitly to prevent version resolution issues
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss=tf.keras.losses.Huber())
model.summary()

### Step 6: Train the Model

We will now train the network on our generated sequences. We keep track of both training and validation losses to ensure the model learns correctly without overfitting.

In [ ]:
# Train the model
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=20,
    batch_size=32,
    verbose=1
)

# Plot the training and validation loss curves
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('LSTM Training and Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss (MSE)')
plt.legend()
plt.grid(True)
plt.show()

### Step 7: Evaluate the Model

To see how well our trained model performs, we will compute two common regression evaluation metrics:
* **Mean Absolute Error (MAE)**: Measures the average magnitude of errors in predictions without considering their direction.
* **Root Mean Squared Error (RMSE)**: Calculates the square root of average squared differences, giving higher weight to larger errors.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Generate predictions for the validation set
y_pred = model.predict(X_val)

# Calculate metrics
mae = mean_absolute_error(y_val, y_pred)
mse = mean_squared_error(y_val, y_pred)
rmse = np.sqrt(mse)

print(f"Validation Mean Absolute Error (MAE): {mae:.5f}")
print(f"Validation Root Mean Squared Error (RMSE): {rmse:.5f}")

### Step 8: Visualize Actual vs. Predicted Prices

Let's plot the target validation closing prices against our model's predictions to visually inspect the forecasting accuracy.

In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(y_val, label='Actual Normalized Close Price', color='blue', alpha=0.8)
plt.plot(y_pred, label='Predicted Normalized Close Price', color='red', linestyle='--', alpha=0.9)
plt.title('Actual vs. Predicted Netflix Stock Prices')
plt.xlabel('Days')
plt.ylabel('Normalized Closing Price')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import plotly.graph_objects as go
import pandas as pd
import numpy as np

# 1. Align dates and actual prices for the validation segment
val_dates = df['date'].iloc[split_idx + WINDOW_SIZE:].reset_index(drop=True)
val_df = df.iloc[split_idx + WINDOW_SIZE:].reset_index(drop=True)

# 2. Build the interactive Candlestick chart
fig = go.Figure()

# Add actual candlesticks (Open, High, Low, Close)
fig.add_trace(go.Candlestick(
    x=val_dates,
    open=val_df['open'],
    high=val_df['high'],
    low=val_df['low'],
    close=val_df['close'],
    name='Actual Stock Price (Candles)'
))

# Add our LSTM model's predictions as a line trace
fig.add_trace(go.Scatter(
    x=val_dates,
    y=y_pred_usd,
    mode='lines',
    line=dict(color='blue', width=2, dash='dash'),
    name='LSTM Predicted Close'
))

# Style layout
fig.update_layout(
    title='Netflix Stock Price Candlestick Chart with LSTM Predictions',
    yaxis_title='Stock Price ($)',
    xaxis_title='Date',
    xaxis_rangeslider_visible=True,
    template='plotly_dark'
)

# Display
fig.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Fit the updated model for 50 epochs
print("Training the updated model for 50 epochs...")
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=32,
    verbose=0
)

# 2. Generate predictions on validation set
y_pred_scaled = model.predict(X_val)

# We need to inverse-transform to scale back to actual USD
num_features = len(feature_cols)
dummy_pred = np.zeros((len(y_pred_scaled), num_features))
dummy_pred[:, target_index] = y_pred_scaled.flatten()
y_pred_usd = scaler.inverse_transform(dummy_pred)[:, target_index]

dummy_val = np.zeros((len(y_val), num_features))
dummy_val[:, target_index] = y_val.flatten()
y_val_usd = scaler.inverse_transform(dummy_val)[:, target_index]

# 3. Calculate evaluation metrics in USD
mae = mean_absolute_error(y_val_usd, y_pred_usd)
mse = mean_squared_error(y_val_usd, y_pred_usd)
rmse = np.sqrt(mse)
r2 = r2_score(y_val_usd, y_pred_usd)
mape = np.mean(np.abs((y_val_usd - y_pred_usd) / y_val_usd)) * 100
accuracy_percentage = 100 - mape

print(f"\n--- Evaluation Metrics in USD (50 Epochs) ---")
print(f"Validation Mean Absolute Error (MAE): ${mae:.2f}")
print(f"Validation Root Mean Squared Error (RMSE): ${rmse:.2f}")
print(f"Model Accuracy (100 - MAPE): {accuracy_percentage:.2f}%")
print(f"R-squared (R2) Score: {r2:.4f} ({r2*100:.2f}% variance explained)")

# 4. Plot actual vs predicted prices
plt.figure(figsize=(14, 6))
plt.plot(y_val_usd, label='Actual Stock Price (USD)', color='blue', alpha=0.8)
plt.plot(y_pred_usd, label='Predicted Stock Price (USD)', color='red', linestyle='--', alpha=0.9)
plt.title('Actual vs. Predicted Netflix Stock Prices (50 Epochs)')
plt.xlabel('Days')
plt.ylabel('Stock Price ($)')
plt.legend()
plt.grid(True)
plt.show()

# 5. Predict on Unseen Dates (predicting the next day after the dataset ends)
last_sequence = data_matrix[-WINDOW_SIZE:]
last_sequence_reshaped = np.reshape(last_sequence, (1, WINDOW_SIZE, num_features))

next_pred_scaled = model.predict(last_sequence_reshaped)
dummy_next_pred = np.zeros((1, num_features))
dummy_next_pred[0, target_index] = next_pred_scaled[0, 0]
next_pred_usd = scaler.inverse_transform(dummy_next_pred)[0, target_index]

# Determine the next trading date
last_date = df['date'].iloc[-1]
next_date = last_date + pd.Timedelta(days=1) if last_date.dayofweek < 4 else last_date + pd.Timedelta(days=3)

print(f"\n--- Unseen Future Date Prediction ---")
print(f"Last Date in Dataset: {last_date.strftime('%Y-%m-%d')} (Actual Close: ${df['close'].iloc[-1]:.2f})")
print(f"Predicted Close for Next Trading Day ({next_date.strftime('%Y-%m-%d')}): ${next_pred_usd:.2f}")

In [ ]:
from sklearn.metrics import r2_score

# Calculate R-squared (Coefficient of Determination)
r2 = r2_score(y_val_usd, y_pred_usd)

# Calculate Mean Absolute Percentage Error (MAPE)
mape = np.mean(np.abs((y_val_usd - y_pred_usd) / y_val_usd)) * 100

# Accuracy defined as (100% - MAPE)
accuracy_percentage = 100 - mape

print(f"--- LSTM Regression Accuracy Metrics ---")
print(f"Mean Absolute Percentage Error (MAPE): {mape:.2f}%")
print(f"Model Accuracy (100 - MAPE): {accuracy_percentage:.2f}%")
print(f"R-squared (R2) Score: {r2:.4f} (indicating {r2*100:.2f}% of variance explained)")